In [1]:
import logging
from pyspark.sql import SparkSession
from collections import Counter
from pyspark.sql.functions import sum, round, col, avg, countDistinct, datediff, year, month, dense_rank
from pyspark.sql.window import Window

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger("TradeCorpETL")

spark = SparkSession.builder.appName("TradeCorpETL").getOrCreate()

logger.info(f"Session Spark créée - version {spark.version}")
logger.info("Chargement des fichiers parquet")

tables = [
    "customers",
    "orders",
    "order_details",
    "products",
    "categories",
    "suppliers",
    "employees",
    "shippers"
]

dataframes = {}

for name in tables:
    logger.info(f"Chargement de {name}")
    dataframes[name] = spark.read.parquet(f"../data/tmp/{name}")
    logger.info(f"{name} chargé avec succès")

logger.info("Chargement de tous les fichiers Parquet terminé")

2026-08-31 07:31:22,425 | INFO | NumExpr defaulting to 12 threads.
2026-08-31 07:31:25,929 | INFO | Session Spark créée - version 4.2.0
2026-08-31 07:31:25,929 | INFO | Chargement des fichiers parquet
2026-08-31 07:31:25,930 | INFO | Chargement de customers
2026-08-31 07:31:27,422 | INFO | customers chargé avec succès
2026-08-31 07:31:27,422 | INFO | Chargement de orders
2026-08-31 07:31:27,606 | INFO | orders chargé avec succès
2026-08-31 07:31:27,607 | INFO | Chargement de order_details
2026-08-31 07:31:27,790 | INFO | order_details chargé avec succès
2026-08-31 07:31:27,791 | INFO | Chargement de products
2026-08-31 07:31:27,953 | INFO | products chargé avec succès
2026-08-31 07:31:27,954 | INFO | Chargement de categories
2026-08-31 07:31:28,109 | INFO | categories chargé avec succès
2026-08-31 07:31:28,109 | INFO | Chargement de suppliers
2026-08-31 07:31:28,262 | INFO | suppliers chargé avec succès
2026-08-31 07:31:28,262 | INFO | Chargement de employees
2026-08-31 07:31:28,413 | 

**Question 21** : Joindre df_orders et df_customers sur customer_id. Garder uniquement : order_id, company_name, country,
order_date, freight.

In [2]:
logger.info("Question 21 - Jointure des commandes et des clients")

df_orders_customers = (dataframes["orders"]
    .join(dataframes["customers"], on="customer_id", how="inner")
    .select("order_id","company_name", "country", "order_date", "freight"))

logger.info("Jointure des commandes avec les clients terminée")

logger.info("Question 21 terminée")

2026-08-31 07:31:28,565 | INFO | Question 21 - Jointure des commandes et des clients
2026-08-31 07:31:28,669 | INFO | Jointure des commandes avec les clients terminée
2026-08-31 07:31:28,669 | INFO | Question 21 terminée


In [3]:
df_orders_customers.show(5)

+--------+--------------------+-------+----------+-------+
|order_id|        company_name|country|order_date|freight|
+--------+--------------------+-------+----------+-------+
|   10400|  Eastern Connection|     UK|1997-01-01|  83.93|
|   10401|Rattlesnake Canyo...|    USA|1997-01-01|  12.51|
|   10402|        Ernst Handel|AUSTRIA|1997-01-02|  67.88|
|   10403|        Ernst Handel|AUSTRIA|1997-01-03|  73.79|
|   10404|Magazzini Aliment...|  ITALY|1997-01-03| 155.97|
+--------+--------------------+-------+----------+-------+
only showing top 5 rows


**Question 22** : Joindre df_order_details et df_products sur product_id. Ajouter les colonnes product_name, category_id,
unit_price depuis products.

In [4]:
logger.info("Question 22 - Jointure du détail des commandes et des produits")

df_products_order_details = (dataframes["order_details"]
    .join(dataframes["products"].select("product_id", "product_name", "category_id", "unit_price"), on="product_id", how="inner"))

logger.info("Jointure du détail des commandes et des produits terminée")
logger.info("Question 22 terminée")

2026-08-31 07:31:30,410 | INFO | Question 22 - Jointure du détail des commandes et des produits
2026-08-31 07:31:30,451 | INFO | Jointure du détail des commandes et des produits terminée
2026-08-31 07:31:30,452 | INFO | Question 22 terminée


In [5]:
df_products_order_details.show(5)

+----------+--------+-------------+--------+--------+----------+--------------------+-----------+----------+
|product_id|order_id|prix_unitaire|quantite|discount|sous_total|        product_name|category_id|unit_price|
+----------+--------+-------------+--------+--------+----------+--------------------+-----------+----------+
|        11|   10248|         14.0|      12|     0.0|     168.0|      Queso Cabrales|          4|      21.0|
|        72|   10248|         34.8|       5|     0.0|     174.0|Mozzarella di Gio...|          4|      34.8|
|        14|   10249|         18.6|       9|     0.0|     167.4|                Tofu|          7|     23.25|
|        51|   10249|         42.4|      40|     0.0|    1696.0|Manjimup Dried Ap...|          7|      53.0|
|        41|   10250|          7.7|      10|     0.0|      77.0|Jack's New Englan...|          8|      9.65|
+----------+--------+-------------+--------+--------+----------+--------------------+-----------+----------+
only showing top 5 

**Question 23** : Joindre df_products et df_categories sur category_id pour enrichir chaque produit avec category_name et
description.

In [6]:
logger.info("Question 23 - Jointure de produit et des catégories")

df_products_categories = (dataframes["products"]
    .join(dataframes["categories"].select("category_id", "category_name", "description"), on="category_id", how="inner"))

logger.info("Jointure de produit et des catégories terminée")
logger.info("Question 23 terminée")

2026-08-31 07:31:30,882 | INFO | Question 23 - Jointure de produit et des catégories
2026-08-31 07:31:30,914 | INFO | Jointure de produit et des catégories terminée
2026-08-31 07:31:30,914 | INFO | Question 23 terminée


In [7]:
df_products_categories.show(5)

+-----------+----------+--------------------+-----------+-------------------+----------+--------------+--------------+-------------+------------+--------+-------------+--------------------+
|category_id|product_id|        product_name|supplier_id|  quantity_per_unit|unit_price|units_in_stock|units_on_order|reorder_level|discontinued|en_stock|category_name|         description|
+-----------+----------+--------------------+-----------+-------------------+----------+--------------+--------------+-------------+------------+--------+-------------+--------------------+
|          2|         3|       Aniseed Syrup|          1|12 - 550 ml bottles|      10.0|            13|            70|           25|           0|    true|   Condiments|Sweet and savory ...|
|          2|         4|Chef Anton's Caju...|          2|     48 - 6 oz jars|      22.0|            53|             0|            0|           0|    true|   Condiments|Sweet and savory ...|
|          2|         6|Grandma's Boysenb...|     

**Question 24** : 
A - Réaliser une première jointure complète (order_details, orders, customers, products enrichi avec
categories, employees, shippers) sans renommer aucune colonne. Lister ensuite les colonnes qui apparaissent
en double grâce à Counter.

In [8]:
logger.info("Question 24 A - Jointure complète sans renommer")

df_complet = (
    dataframes["order_details"]
    .join(dataframes["orders"], on="order_id", how="inner")
    .join(dataframes["customers"], on="customer_id", how="inner")
    .join(df_products_categories, on="product_id", how="inner")
    .join(dataframes["employees"], on="employee_id", how="inner")
    .join(dataframes["shippers"], on="shipper_id", how="inner")
)
compteur = Counter(df_complet.columns)

for colonne, nombre in compteur.items():
    if nombre > 1:
        logger.warning(f"Colonne en double : {colonne} ({nombre} fois)")
        
logger.info("Question 24 A terminée")

2026-08-31 07:31:31,289 | INFO | Question 24 A - Jointure complète sans renommer
2026-08-31 07:31:31,358 | WARNING | Colonne en double : company_name (2 fois)
2026-08-31 07:31:31,358 | WARNING | Colonne en double : city (2 fois)
2026-08-31 07:31:31,359 | WARNING | Colonne en double : country (2 fois)
2026-08-31 07:31:31,359 | WARNING | Colonne en double : phone (2 fois)
2026-08-31 07:31:31,360 | INFO | Question 24 A terminée


B - Pour chaque colonne identifiée en Q24a, la renommer dans sa table d'origine avant de refaire la jointure,
en la préfixant selon la table (customer_country, employee_country, shipper_name...). Reconstruire ensuite
df_orders_enriched avec ces tables renommées, puis vérifier qu'il ne reste plus aucun doublon.

In [9]:
logger.info("Question 24 B - Renommage des colonnes en double")

df_customers_renamed = (
    dataframes["customers"]
    .withColumnRenamed("company_name", "customer_company_name")
    .withColumnRenamed("country", "customer_country")
    .withColumnRenamed("city", "customer_city")
    .withColumnRenamed("phone", "customer_phone")
)

df_employees_renamed = (
    dataframes["employees"]
    .withColumnRenamed("country", "employee_country")
    .withColumnRenamed("city", "employee_city")
)

df_shippers_renamed = (
    dataframes["shippers"]
    .withColumnRenamed("compagny_name", "shipper_company_name")
    .withColumnRenamed("phone", "shipper_phone")
)

df_orders_enriched = (
    dataframes["order_details"]
    .join(dataframes["orders"], on="order_id", how="inner")
    .join(df_customers_renamed, on="customer_id", how="inner")
    .join(df_products_categories, on="product_id", how="inner")
    .join(df_employees_renamed, on="employee_id", how="inner")
    .join(df_shippers_renamed, on="shipper_id", how="inner")
)

compteur = Counter(df_orders_enriched.columns)

doublon_trouve = False

for colonne, nombre in compteur.items():
    if nombre > 1:
        logger.warning(f"Colonne en double : {colonne} ({nombre} fois)")
        doublon_trouve = True

if not doublon_trouve:
    logger.info("Aucune colonne en double")

logger.info("Question 24 B terminée")

2026-08-31 07:31:31,368 | INFO | Question 24 B - Renommage des colonnes en double
2026-08-31 07:31:31,462 | INFO | Aucune colonne en double
2026-08-31 07:31:31,462 | INFO | Question 24 B terminée


**Question 25** : Calculer le chiffre d'affaires total par client (company_name) depuis df_orders_enriched. Trier par CA
décroissant. Afficher le top 10.

In [10]:
logger.info("Question 25 - Calcul du chiffre d'affaires total par client")

df_ca_customers = (
    df_orders_enriched
        .groupBy("customer_company_name")
        .agg(round(sum("sous_total"), 2).alias("chiffre_affaires"))
        .orderBy(col("chiffre_affaires").desc())
)

logger.info("Top 10 des clients par CA")
df_ca_customers.show(10)

logger.info("question 25 terminée")

2026-08-31 07:31:31,468 | INFO | Question 25 - Calcul du chiffre d'affaires total par client
2026-08-31 07:31:31,567 | INFO | Top 10 des clients par CA
2026-08-31 07:31:33,009 | INFO | question 25 terminée


+---------------------+----------------+
|customer_company_name|chiffre_affaires|
+---------------------+----------------+
|           QUICK-Stop|        51682.74|
|   Save-a-lot Markets|        40238.09|
|         Ernst Handel|        39975.91|
|       Mère Paillarde|        22871.07|
| Rattlesnake Canyo...|         17636.1|
|        Simons bistro|        16232.42|
| Hungry Owl All-Ni...|        14403.03|
|       Folk och fä HB|        13200.92|
|     HILARION-Abastos|        11799.74|
|   Berglunds snabbköp|        11758.92|
+---------------------+----------------+
only showing top 10 rows


**Question 26** : Calculer le CA total par catégorie de produits. Afficher le nombre de produits distincts vendus par catégorie.

In [11]:
logger.info("Question 26 - Calcul du chiffre d'affaires par catégorie")

df_ca_categories = (
    df_orders_enriched
    .groupBy("category_name")
    .agg(
        round(sum("sous_total"), 2).alias("chiffre_affaires"),
        countDistinct("product_id").alias("nb_produits_distincts")
    )
    .orderBy(col("chiffre_affaires").desc())
)

df_ca_categories.show()

logger.info("Question 26 terminée")

2026-08-31 07:31:33,019 | INFO | Question 26 - Calcul du chiffre d'affaires par catégorie
2026-08-31 07:31:34,129 | INFO | Question 26 terminée


+--------------+----------------+---------------------+
| category_name|chiffre_affaires|nb_produits_distincts|
+--------------+----------------+---------------------+
|Dairy Products|        108086.9|                    9|
|     Beverages|        90368.64|                    9|
|   Confections|        82657.78|                   13|
|       Seafood|        66959.23|                   12|
|    Condiments|         54995.0|                   11|
|Grains/Cereals|        51463.63|                    6|
|       Produce|        40992.09|                    4|
|  Meat/Poultry|        11017.17|                    2|
+--------------+----------------+---------------------+



**Question 27** : Calculer le CA mensuel. Utiliser date_trunc ou month() et year() pour extraire le mois et l'année.

In [12]:
logger.info("Question 27 - Calcul du chiffre d'affaires mensuel")

df_ca_mensuel = (
    df_orders_enriched
    .groupBy(
        year("order_date").alias("annee"),
        month("order_date").alias("mois")
    )
    .agg(round(sum("sous_total"), 2).alias("chiffre_affaires"))
    .orderBy("annee", "mois")
)

df_ca_mensuel.show()

logger.info("Question 27 terminée")

2026-08-31 07:31:34,138 | INFO | Question 27 - Calcul du chiffre d'affaires mensuel
2026-08-31 07:31:34,852 | INFO | Question 27 terminée


+-----+----+----------------+
|annee|mois|chiffre_affaires|
+-----+----+----------------+
| 1997|   1|        51487.51|
| 1997|   2|        31549.04|
| 1997|   3|        33226.33|
| 1997|   4|         41510.6|
| 1997|   5|        48895.27|
| 1997|   6|        29875.47|
| 1997|   7|        45162.88|
| 1997|   8|        38039.93|
| 1997|   9|        43335.43|
| 1997|  10|         48574.5|
| 1997|  11|        39898.78|
| 1997|  12|         54984.7|
+-----+----+----------------+



**Question 28** : Calculer pour chaque employé (full_name) : le nombre de commandes traitées, le CA total généré et le délai
moyen de livraison en jours.

In [13]:
logger.info("Question 28 - Calcul du CA généré, délai moyen et nb de commandes traitées pour chaque employé")

df_performance_employes = (
    df_orders_enriched
    .groupBy("employee_id", "full_name")
    .agg(
        round(sum("sous_total"), 2).alias("chiffre_affaires"),
        round(avg(datediff("shipped_date", "order_date")), 1).alias("delai_moyen_jours"),
        countDistinct("order_id").alias("nb_commandes")
    )
    .orderBy(col("chiffre_affaires").desc())
)

df_performance_employes.show()

logger.info("Question 28 terminée")

2026-08-31 07:31:34,860 | INFO | Question 28 - Calcul du CA généré, délai moyen et nb de commandes traitées pour chaque employé
2026-08-31 07:31:35,874 | INFO | Question 28 terminée


+-----------+----------------+----------------+-----------------+------------+
|employee_id|       full_name|chiffre_affaires|delai_moyen_jours|nb_commandes|
+-----------+----------------+----------------+-----------------+------------+
|          4|Margaret Peacock|       104193.78|              8.3|          75|
|          3| Janet Leverling|        97081.27|              8.9|          71|
|          1|   Nancy Davolio|        81898.92|              7.8|          54|
|          2|   Andrew Fuller|        54907.03|             10.2|          40|
|          7|     Robert King|        49562.78|              9.8|          33|
|          8|  Laura Callahan|        47077.95|              8.0|          53|
|          6|  Michael Suyama|        34037.07|              7.9|          33|
|          9|  Anne Dodsworth|        20595.99|             10.0|          18|
|          5| Steven Buchanan|        17185.65|              6.5|          18|
+-----------+----------------+----------------+-----

**Question 29** : Classer les produits par CA généré avec dense_rank(). Utiliser une Window partitionnée par category_name.

In [14]:
logger.info("Question 29 - Classer les produits par CA généré")

df_ca_produits = (
    df_orders_enriched
    .groupBy("category_name", "product_id","product_name")
    .agg(round(sum("sous_total"), 2).alias("chiffre_affaires"))
)

window_category = (Window.partitionBy("category_name").orderBy(col("chiffre_affaires").desc()))

df_classement_produits = (
    df_ca_produits
    .withColumn("rang",dense_rank().over(window_category))
    .orderBy("category_name", "rang")
)

df_classement_produits.show()

logger.info("Question 29 terminée")


2026-08-31 07:31:35,882 | INFO | Question 29 - Classer les produits par CA généré
2026-08-31 07:31:36,779 | INFO | Question 29 terminée


+-------------+----------+--------------------+----------------+----+
|category_name|product_id|        product_name|chiffre_affaires|rang|
+-------------+----------+--------------------+----------------+----+
|    Beverages|        38|       Côte de Blaye|        49198.09|   1|
|    Beverages|        43|         Ipoh Coffee|         11069.9|   2|
|    Beverages|        76|        Lakkalikööri|          7379.1|   3|
|    Beverages|        70|       Outback Lager|          5468.4|   4|
|    Beverages|        35|      Steeleye Stout|          5274.9|   5|
|    Beverages|        75|Rhönbräu Klosterbier|         4485.55|   6|
|    Beverages|        39|    Chartreuse verte|          4475.7|   7|
|    Beverages|        34|       Sasquatch Ale|          2107.0|   8|
|    Beverages|        67|Laughing Lumberja...|           910.0|   9|
|   Condiments|        65|Louisiana Fiery H...|          9373.2|   1|
|   Condiments|        61|      Sirop d'érable|          9091.5|   2|
|   Condiments|     

**Question 30** : Calculer le CA cumulé par mois (ordre chronologique) avec sum() sur une Window orderBy date.

In [15]:
logger.info("Question 30 - Calcul du CA cumulé par mois")

window_mois = (Window.orderBy("annee", "mois").rowsBetween(Window.unboundedPreceding, Window.currentRow))

df_ca_cumule = (
    df_ca_mensuel
    .withColumn("ca_cumule",round(sum("chiffre_affaires").over(window_mois), 2))
    .orderBy("annee", "mois")
)

df_ca_cumule.show()

logger.info("Question 30 terminée")

2026-08-31 07:39:20,238 | INFO | Question 30 - Calcul du CA cumulé par mois
2026-08-31 07:39:21,016 | INFO | Question 30 terminée


+-----+----+----------------+---------+
|annee|mois|chiffre_affaires|ca_cumule|
+-----+----+----------------+---------+
| 1997|   1|        51487.51| 51487.51|
| 1997|   2|        31549.04| 83036.55|
| 1997|   3|        33226.33|116262.88|
| 1997|   4|         41510.6|157773.48|
| 1997|   5|        48895.27|206668.75|
| 1997|   6|        29875.47|236544.22|
| 1997|   7|        45162.88| 281707.1|
| 1997|   8|        38039.93|319747.03|
| 1997|   9|        43335.43|363082.46|
| 1997|  10|         48574.5|411656.96|
| 1997|  11|        39898.78|451555.74|
| 1997|  12|         54984.7|506540.44|
+-----+----+----------------+---------+



**Question 31** : Afficher les 5 produits les plus vendus en quantité (toutes commandes confondues).
Afficher les 3 pays clients (customer_country) qui génèrent le plus de chiffre d'affaires.

In [21]:
logger.info("Question 31 - Top 5 des produits les plus vendus en quantité")

df_top_produits = (
    df_orders_enriched
    .groupBy("product_id","product_name")
    .agg(sum("quantite").alias("quantite_totale_vendue"))
    .orderBy(col("quantite_totale_vendue").desc())
)

df_top_produits.show(5)

2026-08-31 07:51:35,959 | INFO | Question 31 - Top 5 des produits les plus vendus en quantité


+----------+--------------------+----------------------+
|product_id|        product_name|quantite_totale_vendue|
+----------+--------------------+----------------------+
|        56|Gnocchi di nonna ...|                   971|
|        59|Raclette Courdavault|                   752|
|        60|   Camembert Pierrot|                   665|
|        75|Rhönbräu Klosterbier|                   630|
|        21| Sir Rodney's Scones|                   610|
+----------+--------------------+----------------------+
only showing top 5 rows


In [22]:
logger.info("Question 31 - Top 3 des pays clients par chiffre d'affaires")

df_top_pays = (
    df_orders_enriched
    .groupBy("customer_country")
    .agg(round(sum("sous_total"), 2).alias("chiffre_affaires"))
    .orderBy(col("chiffre_affaires").desc())
)

df_top_pays.show(3)

logger.info("Question 31 terminée")

2026-08-31 07:51:57,765 | INFO | Question 31 - Top 3 des pays clients par chiffre d'affaires
2026-08-31 07:51:58,171 | INFO | Question 31 terminée


+----------------+----------------+
|customer_country|chiffre_affaires|
+----------------+----------------+
|         GERMANY|       100641.29|
|             USA|        90731.71|
|         AUSTRIA|        46559.49|
+----------------+----------------+
only showing top 3 rows


**Question 32** : Écrire df_orders_enriched en format Parquet dans /home/jovyan/data/output/orders_enriched.parquet. Utiliser le
mode overwrite.

In [24]:
logger.info("Début de l'exportation en parquet")
logger.info("Export du DataFrame df_orders_enriched au format Parquet")

df_orders_enriched.write.mode("overwrite").parquet(f"../data/output/orders_enriched")

logger.info("Export du DataFrame terminé")

2026-08-31 07:56:18,603 | INFO | Début de l'exportation en parquet
2026-08-31 07:56:18,605 | INFO | Export du DataFrame df_orders_enriched au format Parquet
2026-08-31 07:56:19,808 | INFO | Export du DataFrame terminé


In [31]:
count_enriched = df_orders_enriched.count()
logger.info(f"Nombre de lignes dans df_orders_enriched : {count_enriched}")

df = spark.read.parquet("../data/output/orders_enriched")
count_parquet = df.count()
logger.info(f"Nombre de lignes dans le Parquet : {count_parquet}")

if count_enriched == count_parquet:
    logger.info("Les deux DataFrames ont le même nombre de lignes.")
else:
    logger.warning(f"Il n'y a pas le même nombre de lignes entre les deux DataFrames !!!")

2026-08-31 08:04:39,713 | INFO | Nombre de lignes dans df_orders_enriched : 893
2026-08-31 08:04:39,949 | INFO | Nombre de lignes dans le Parquet : 893
2026-08-31 08:04:39,950 | INFO | Les deux DataFrames ont le même nombre de lignes.
